In [1]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import os
import torch
from trl import SFTTrainer
from datasets import load_dataset
from transformers import TrainingArguments


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "google/gemma-2b",
    max_seq_length = 512,
    load_in_4bit = True,
    dtype = torch.float16
)

==((====))==  Unsloth 2026.9.12: Fast Gemma patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

Unsloth: Will load unsloth/gemma-2b-bnb-4bit as a legacy tokenizer.


In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj",],
)

Unsloth 2026.9.12 patched 18 layers with 18 QKV layers, 18 O layers and 18 MLP layers.


In [4]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma"
)
dataset = load_dataset("Abirate/english_quotes", split="train")

Unsloth: Restored added_tokens_decoder metadata in /content/_unsloth_sentencepiece_temp/tokenizer_6gw7yamv/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/_unsloth_sentencepiece_temp/tokenizer_6gw7yamv.


In [5]:
def formatting_prompts_func(examples):
    quotes = examples["quote"]
    authors = examples["author"]
    tags = examples["tags"]

    texts = []
    for quote, author, tag_list in zip(quotes, authors, tags):
        # Grab up to 3 tags to keep the user prompt clean
        tag_str = ", ".join(tag_list[:3]) if tag_list else "general"

        # Build the conversation using "assistant" instead of "model"
        messages = [
            {"role": "user", "content": f"Share a quote by {author} about {tag_str}."},
            {"role": "assistant", "content": quote}
        ]

        # apply_chat_template automatically adds <start_of_turn> tags and <eos>
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)

    return { "text": texts }

# 4. Apply the formatting to the dataset
dataset = dataset.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [6]:
dataset[1]["text"]

"<bos><start_of_turn>user\nShare a quote by Marilyn Monroe about best, life, love.<end_of_turn>\n<start_of_turn>model\n“I'm selfish, impatient and a little insecure. I make mistakes, I am out of control and at times hard to handle. But if you can't handle me at my worst, then you sure as hell don't deserve me at my best.”<end_of_turn>\n"

In [7]:
dataset

Dataset({
    features: ['quote', 'author', 'tags', 'text'],
    num_rows: 2508
})

In [13]:
from trl import SFTConfig

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,

    args = SFTConfig(
        dataset_text_field = "text",
        max_length = 512,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        fp16 = True,
        bf16 = False,
        logging_steps = 50,
        output_dir = "outputs",
        optim = "paged_adamw_8bit",
        lr_scheduler_type = "cosine",
        seed = 3407,
    ),
)

Unsloth: We found double BOS tokens - we shall remove one automatically.
🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [14]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,508 | Num Epochs = 2 | Total steps = 628
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 19,611,648 of 2,525,784,064 (0.78% trained)


Step,Training Loss
50,1.314491
100,1.297292
150,1.476262
200,1.494030
250,1.577704
300,1.670826
350,1.339238
400,1.282988
450,1.241039
500,1.304449


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-628/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-628.


TrainOutput(global_step=628, training_loss=1.367684090972706, metrics={'train_runtime': 1269.9787, 'train_samples_per_second': 3.95, 'train_steps_per_second': 0.494, 'total_flos': 3876305433133056.0, 'train_loss': 1.367684090972706, 'epoch': 2.0})

In [15]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GemmaForCausalLM(
      (model): GemmaModel(
        (embed_tokens): GemmaTextScaledWordEmbedding(256000, 2048, padding_idx=0)
        (layers): ModuleList(
          (0-17): 18 x GemmaDecoderLayer(
            (self_attn): GemmaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
             

In [20]:
author = "John Keats"
tags ="art,life,pain"

messages = [
    {"role": "user", "content": f"Share a quote by {author} about {tags}."}
]

In [23]:
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens =True)

print(f"--- Generating quote for {author} ---")
_ = model.generate(
    input_ids=inputs,
    streamer=text_streamer,
    max_new_tokens=128,
    use_cache=True,
    temperature=0,
)

Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Generating quote for John Keats ---
“Do you not see how necessary a world of pain is to school a world of joy?”
